In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.io import read_image
import torchvision.transforms.functional as F_t
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets.utils import download_url
import matplotlib.pyplot as plt
from IPython import display
import cv2
from PIL import Image


try:
    from einops import rearrange
except ImportError:
    %pip install einops
    from einops import rearrange

In [ ]:
class RFF(nn.Module):
    def __init__(self, in_features, out_features, gamma):
        super().__init__()
        self.B = torch.randn(in_features, out_features)*gamma

    def forward(self, x):
        return torch.hstack([torch.sin(torch.matmul(2*math.pi*x, self.B)), torch.cos(torch.matmul(2*math.pi*x, self.B))])

In [ ]:
class RFF_SR(nn.Module):
    def __init__(self, in_features, fourier_features, out_features, gamma):
        super().__init__()
        self.net = nn.Sequential(RFF(in_features, fourier_features, gamma), nn.Linear(2*fourier_features, out_features))

    def forward(self, x):
      return self.net(x)

In [ ]:
def create_coord_map(image_path):
        image = read_image(image_path)
        h, w = image.shape[1], image.shape[2]
        image = image.float()/255
        image = image.permute(1, 2, 0)
        # print(image)

        h_coords = torch.linspace(0, 1, steps=h)
        w_coords = torch.linspace(0, 1, steps=w)
        grid = torch.stack(torch.meshgrid(h_coords, w_coords), dim=-1)
        # print(h, w)

        return grid, image

In [ ]:
image_path = "dog.jpg"
display.Image(image_path)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

grid, image = create_coord_map(image_path)
# print(grid.shape, image.shape)
grid, image = grid.squeeze().to(device), image.squeeze().to(device)

In [ ]:
# Create a 300x300 tensor with all values set to True
boolean_array = torch.full((300, 300), False, dtype=torch.bool)

# Generate 900 random indices
indices = torch.randperm(300*300)[:900]

# Set the randomly selected indices to False
boolean_array.view(-1)[indices] = True

boolean_array.sum()

In [ ]:

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 300)
y = torch.linspace(-1, 1, 300)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {900} random pixels")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 1000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {900} random pixels")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {900} random pixels")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {900} random pixels")
plt.show()


In [ ]:
# Create a 300x300 tensor with all values set to True
boolean_array = torch.full((300, 300), False, dtype=torch.bool)

boolean_array[100:130, 150:180] = True

boolean_array.sum()

In [ ]:

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 300)
y = torch.linspace(-1, 1, 300)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {900} random pixels")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 1000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 201):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {900} random pixels")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {900} random pixels")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {900} random pixels")
plt.show()
